In [2]:
from dofusapi import DofusAPI
from dataprocessor import DataProcessor
# from visualizer import EquipmentVisualizer
from utils import CacheManager
from collections import defaultdict
import numpy as np

In [3]:
equipments = DofusAPI.get_all_equipments()
processor = DataProcessor(equipments)
from visualizer import EquipmentVisualizer
viz = EquipmentVisualizer()
resource_info_getter = CacheManager().get_resource_info

✅ 271 équipements récupérés avec succès


In [4]:
greedy_groups = processor.greedy_grouping(exploration_depth=3)
viz.generate_visualizations(greedy_groups, resource_info_getter)

Starting greedy exploration with min_shared=2, depth=3
Greedy exploration found 21 groups
Group sizes: [40, 2, 40, 10, 11, 18, 14, 42, 3, 2, 6, 3, 6, 2, 2, 4, 3, 9, 8, 2, 2]
Total equipment: 271, Grouped: 271, Ungrouped: 0
Found 21 communities.


Processing communities:   0%|          | 0/21 [00:00<?, ?it/s]

Greedy grouping produced 28 final groups
🎯 Generating reports for 28 groups...

🎉 Groups visualization generated !
👉 Open http://localhost:8000/index.html in your browser


In [24]:
def resource_chain_exploration(equipments, min_chain_length=3):
    """
    Find equipment connected by chains of shared resources
    """
    from collections import defaultdict, deque
    import numpy as np
    
    # Build mappings
    equipment_to_resources = {}
    resource_to_equipments = defaultdict(list)
    
    for equipment in equipments:
        resource_ids = [r.resource_id for r in equipment.recipe]
        equipment_to_resources[equipment.ankama_id] = set(resource_ids)
        for rid in resource_ids:
            resource_to_equipments[rid].append(equipment.ankama_id)
    
    print(f"Resource chain exploration with min_chain={min_chain_length}")
    
    communities = defaultdict(list)
    visited_equipment = set()
    group_id = 0
    
    for start_equipment in equipment_to_resources.keys():
        if start_equipment in visited_equipment:
            continue
            
        # Find all equipment connected through resource chains
        connected_component = set()
        queue = deque([start_equipment])
        visited_equipment.add(start_equipment)
        
        while queue:
            current_eq = queue.popleft()
            connected_component.add(current_eq)
            
            # Get all resources for current equipment
            current_resources = equipment_to_resources[current_eq]
            
            # For each resource, get all equipment that use it
            for resource in current_resources:
                for neighbor_eq in resource_to_equipments[resource]:
                    if neighbor_eq not in visited_equipment:
                        visited_equipment.add(neighbor_eq)
                        queue.append(neighbor_eq)
        
        # Only keep components that meet the minimum chain length
        if len(connected_component) >= min_chain_length:
            communities[np.int32(group_id)] = list(connected_component)
            group_id += 1
        else:
            # Add as individual equipment
            for eq in connected_component:
                communities[np.int32(group_id)] = [eq]
                group_id += 1
    
    print(f"Resource chains found {len([g for g in communities.values() if len(g) > 1])} multi-equipment groups")
    return communities

resource_chain_communities = resource_chain_exploration(equipments, min_chain_length=1)
resource_chain_groups = processor.map_communities_inclusive(resource_chain_communities)
print(f"Final groups after mapping: {len(resource_chain_groups)}")

Resource chain exploration with min_chain=1
Resource chains found 2 multi-equipment groups
Found 4 communities.


Processing communities:   0%|          | 0/4 [00:00<?, ?it/s]

Final groups after mapping: 12


In [ ]:
# processor.diagnose_equipment_exclusion(resolution_range=(0, 20, 0.1))
# groups = processor.find_bi_louvain_groups(resolution_range=(5, 50, 0.1),)
# groups = processor.find_equipment_groups(
#     min_shared_ratio=0.2, 
#     resolution_range=(10, 100, 1),
# )

In [ ]:
grouped_equipments = set()
for grp in groups:
    for equip in grp['equipments']:
        grouped_equipments.add(equip.ankama_id)
        
all_equipments = set(equip.ankama_id for equip in equipments)
ungrouped_equipments = all_equipments - grouped_equipments
print(f"Ungrouped equipments: {len(ungrouped_equipments)}")

In [ ]:

viz.generate_visualizations(groups, resource_info_getter)

🎯 Generating reports for 28 groups...

🎉 Groups visualization generated !
👉 Open http://localhost:8000/index.html in your browser
